# Post-training attention and probe visualizations

Training logs numerical attention activations and probe scalars at full cadence. This notebook downloads exact W&B history steps and recreates the former images locally. Downloaded tables are cached under `notebooks/.cache/`; optional exports go to the gitignored `notebooks/figures/` directory. Each selected step opens a scrollable **All plots** row; switch to **One plot** to browse with left/right arrows. Rendered galleries remain in memory until the kernel restarts.

In [ ]:
import importlib
from pathlib import Path

import ipywidgets as widgets
import wandb
from IPython.display import display
from tqdm.auto import tqdm

from notebooks import interactive as interactive_helpers
from src.analysis import training_visualizations as training_plots
importlib.reload(interactive_helpers)
importlib.reload(training_plots)

from notebooks.interactive import (
    fetch_logged_rows, figure_gallery_widget, prepare_figure_gallery,
    show_layer_step_selector, show_probe_selector,
    show_probe_snapshot_selector, single_figure_widget,
)
from notebooks.utils import (
    DEFAULT_ENTITY, DEFAULT_PROJECT, attention_table_key, fetch_attention_tables,
    fetch_run_config, probe_metric_keys, value_alignment_from_metrics,
    value_alignment_keys,
)
from src.analysis.training_visualizations import (
    attention_table_to_array, plot_attention_alignment, plot_attention_heatmaps,
    plot_probe_overview, plot_probe_slice, plot_probe_trajectory,
    plot_value_alignment, probe_layout_from_config, save_figures,
)

## Select a run

`weights` in the W&B table key means attention activations, not learned parameter weights. Validation is the default because attention dropout is disabled there. Each plot below discovers all available logged steps and initially shows the latest one.

In [ ]:
ENTITY = "okyksl"
PROJECT = "hmm-attention"
RUN_ID = "ex75hkb0"
SPLIT = "val"
SAVE_FIGURES = False
OUTPUT_DIR = Path("notebooks/figures") / RUN_ID

api = wandb.Api()
run = api.run(f"{ENTITY}/{PROJECT}/{RUN_ID}")
config = fetch_run_config(RUN_ID, entity=ENTITY, project=PROJECT, api=api)
teacher_cfg = config["teacher"]
student_cfg = config["student"]


tags = config.get("misc", {}).get("wandb", {}).get("tags", [])
print(f"{run.name} ({run.id}) — tags: {tags}")

## Attention heatmaps and span alignment

Select any attention layer and any exact step logged for that layer.

In [ ]:
attention_layers = [
    f"L{index + 1}" for index in range(int(student_cfg["num_blocks"]))
]
attention_keys = {
    layer: attention_table_key(layer, SPLIT) for layer in attention_layers
}
attention_rows = fetch_logged_rows(
    run, list(attention_keys.values()), "Scanning attention layers and steps"
)
attention_steps_by_layer = {
    layer: [
        step for step, row in sorted(attention_rows.items())
        if row.get(key) is not None
    ]
    for layer, key in attention_keys.items()
}
attention_tables = {}
attention_gallery_cache = {}
span_lengths = teacher_cfg.get("span_lengths")
stride = teacher_cfg.get("stride")
context_length = None
if span_lengths:
    context_length = ((len(span_lengths) - 1) * stride + span_lengths[-1]
                      if stride is not None else sum(span_lengths))

def render_attention(layer, step):
    layer = str(layer)
    step = int(step)
    selection = (layer, step)
    if selection in attention_gallery_cache:
        return figure_gallery_widget(attention_gallery_cache[selection])

    stage_count = 4 + int(context_length is not None) + int(SAVE_FIGURES)
    with tqdm(
        total=stage_count, desc=f"Attention {layer}, step {step}",
        unit="stage",
        leave=False, dynamic_ncols=True,
    ) as progress:
        progress.set_postfix_str("loading table")
        if selection not in attention_tables:
            attention_tables[selection] = fetch_attention_tables(
                RUN_ID, [step], layer=layer, split=SPLIT,
                entity=ENTITY, project=PROJECT, api=api,
            )[step]
        table = attention_tables[selection]
        progress.update()

        progress.set_postfix_str("building heatmaps")
        attention = attention_table_to_array(table)
        figures = plot_attention_heatmaps(attention, step=step, split=SPLIT)
        progress.update()

        if context_length is not None:
            progress.set_postfix_str("building alignment plots")
            figures.update(plot_attention_alignment(
                attention, list(span_lengths), context_length, step, SPLIT, stride
            ))
            progress.update()
        if SAVE_FIGURES:
            progress.set_postfix_str("saving figures")
            save_figures(
                figures, OUTPUT_DIR / "attention",
                f"{layer}_{SPLIT}_step{step}",
            )
            progress.update()

        progress.set_postfix_str("preparing gallery")
        attention_gallery_cache[selection] = prepare_figure_gallery(figures)
        progress.update()

        progress.set_postfix_str("preparing view")
        progress.update()
    return figure_gallery_widget(attention_gallery_cache[selection])


attention_control = show_layer_step_selector(
    attention_steps_by_layer, render_attention,
    f"No attention tables are logged for split {SPLIT!r}.",
)

## Value-projection alignment

This section applies only to runs with `attention_disentanglement=true`. The heatmaps are reconstructed from numerical norm and cosine histories.

In [ ]:
if student_cfg.get("attention_disentanglement", False) and "span_lengths" in teacher_cfg:
    num_heads = int(student_cfg["num_heads"])
    num_teacher = int(teacher_cfg.get("window", len(teacher_cfg["span_lengths"])))
    value_keys_by_layer = {}
    for layer in attention_layers:
        norm_keys, cosine_keys = value_alignment_keys(
            num_heads, num_teacher, layer, SPLIT
        )
        value_keys_by_layer[layer] = norm_keys + cosine_keys
    value_keys = [
        key for keys_for_layer in value_keys_by_layer.values()
        for key in keys_for_layer
    ]
    value_rows = fetch_logged_rows(
        run, value_keys, "Scanning value-alignment steps"
    )
    value_steps_by_layer = {
        layer: [
            step for step, row in sorted(value_rows.items())
            if all(row.get(key) is not None for key in keys_for_layer)
        ]
        for layer, keys_for_layer in value_keys_by_layer.items()
    }
    value_gallery_cache = {}

    def render_value_alignment(layer, step):
        layer = str(layer)
        step = int(step)
        selection = (layer, step)
        if selection in value_gallery_cache:
            return figure_gallery_widget(value_gallery_cache[selection])

        stage_count = 4 + int(SAVE_FIGURES)
        with tqdm(
            total=stage_count,
            desc=f"Value alignment {layer}, step {step}", unit="stage",
            leave=False, dynamic_ncols=True,
        ) as progress:
            row = value_rows[step]
            norms, cosine = value_alignment_from_metrics(
                row, num_heads, num_teacher, layer, SPLIT
            )
            progress.update()

            progress.set_postfix_str("building plots")
            figures = plot_value_alignment(norms, cosine, step, SPLIT)
            progress.update()
            if SAVE_FIGURES:
                progress.set_postfix_str("saving figures")
                save_figures(
                    figures, OUTPUT_DIR / "value",
                    f"{layer}_{SPLIT}_step{step}",
                )
                progress.update()

            progress.set_postfix_str("preparing gallery")
            value_gallery_cache[selection] = prepare_figure_gallery(figures)
            progress.update()

            progress.set_postfix_str("preparing view")
            progress.update()
        return figure_gallery_widget(value_gallery_cache[selection])

    value_control = show_layer_step_selector(
        value_steps_by_layer, render_value_alignment,
        "No complete per-head value-alignment steps are logged for this run.",
    )
else:
    print("No logged per-head value alignment is available for this run.")

## Probe explorer

Use the **Temporal** tab to fix layer, level, slot, and offset, then plot one selected metric over all logged steps. Use **Snapshot** to fix a training step and choose any two probe dimensions as the heatmap axes, or select **All probes** for a level-by-offset grid containing every layer and slot. Only the active tab is visible. Invalid level/slot cells and unavailable positive-offset excess NLL values remain blank.

In [ ]:
probe_mode = config.get("misc", {}).get("probe", {}).get("mode", "off")
if probe_mode != "off":
    layout = probe_layout_from_config(config)
    requests = [
        (level, offset, metric)
        for level in range(len(layout.slots_per_level))
        for offset in layout.offsets_by_level[level]
        for metric in (("acc",) if offset > 0 else ("acc", "excess_nll"))
    ]
    keys = [
        key
        for level, offset, metric in requests
        for key in probe_metric_keys(
            level, offset, metric, layout.num_layers,
            layout.slots_per_level[level], SPLIT,
        )
    ]
    probe_rows = fetch_logged_rows(run, keys, "Scanning probe steps")
    if probe_rows:
        probe_trajectory_cache = {}

        def render_probe_trajectory(layer, level, slot, offset, metric):
            selection = (
                int(layer), int(level), int(slot), int(offset), str(metric)
            )
            if selection not in probe_trajectory_cache:
                layer, level, slot, offset, metric = selection
                with tqdm(
                    total=2, desc="Preparing probe trajectory", unit="stage",
                    leave=False, dynamic_ncols=True,
                ) as progress:
                    figure = plot_probe_trajectory(
                        probe_rows, layer, level, slot, offset, metric, SPLIT
                    )
                    progress.update()
                    metric_label = (
                        "accuracy" if metric == "acc" else "excess NLL"
                    )
                    name = (
                        f"L{layer} / level{level} / slot{slot} / "
                        f"k={offset:+d} / {metric_label}"
                    )
                    if SAVE_FIGURES:
                        save_figures(
                            {"trajectory": figure}, OUTPUT_DIR / "probe",
                            f"L{layer}_level{level}_slot{slot}_"
                            f"k{offset:+d}_{metric}",
                        )
                    probe_trajectory_cache[selection] = prepare_figure_gallery(
                        {name: figure}
                    )
                    progress.update()
            return single_figure_widget(probe_trajectory_cache[selection])

        temporal_page = widgets.Output()
        with temporal_page:
            probe_control = show_probe_selector(
                layout.num_layers, layout.slots_per_level,
                layout.offsets_by_level,
                render_probe_trajectory,
            )

        probe_snapshot_cache = {}

        def render_probe_snapshot(
            step, view, metric, x_axis, y_axis,
            layer, level, slot, offset,
        ):
            step = int(step)
            coordinates = {
                "layer": int(layer), "level": int(level),
                "slot": int(slot), "offset": int(offset),
            }
            if view == "overview":
                selection = (view, step, str(metric))
            else:
                selection = (
                    view, step, str(metric), str(x_axis), str(y_axis),
                    *coordinates.values(),
                )
            if selection not in probe_snapshot_cache:
                with tqdm(
                    total=2, desc=f"Preparing probe snapshot {step}",
                    unit="stage", leave=False, dynamic_ncols=True,
                ) as progress:
                    if view == "overview":
                        figure = plot_probe_overview(
                            probe_rows[step], step, metric, SPLIT,
                            layout.num_layers, layout.slots_per_level,
                            layout.offsets_by_level,
                        )
                        name = f"All probes / step {step} / {metric}"
                        prefix = f"step{step}_all_{metric}"
                    else:
                        figure = plot_probe_slice(
                            probe_rows[step], step, x_axis, y_axis,
                            coordinates, metric, SPLIT, layout.num_layers,
                            layout.slots_per_level,
                            layout.offsets_by_level,
                        )
                        name = (
                            f"{y_axis} × {x_axis} / step {step} / {metric}"
                        )
                        prefix = (
                            f"step{step}_{y_axis}_by_{x_axis}_{metric}"
                        )
                    progress.update()
                    if SAVE_FIGURES:
                        save_figures(
                            {"snapshot": figure}, OUTPUT_DIR / "probe",
                            prefix,
                        )
                    probe_snapshot_cache[selection] = prepare_figure_gallery(
                        {name: figure}
                    )
                    progress.update()
            return single_figure_widget(probe_snapshot_cache[selection])

        snapshot_page = widgets.Output()
        with snapshot_page:
            probe_snapshot_control = show_probe_snapshot_selector(
                probe_rows, layout.num_layers, layout.slots_per_level,
                layout.offsets_by_level, render_probe_snapshot,
            )
        probe_pages = widgets.Tab(children=(temporal_page, snapshot_page))
        probe_pages.set_title(0, "Temporal")
        probe_pages.set_title(1, "Snapshot")
        probe_pages.selected_index = 0
        display(probe_pages)
    else:
        print("No probe values are logged for this run.")
else:
    print("Probe mode is off for this run.")